# LIDA + StoryWeaver Merge Analysis

Purpose of this notebook:

1. Inspect `lida_stories_en.json` and `storyweaver_texts_cleaned.json`.
2. Compare their structures and available metadata.
3. Understand existing StoryWeaver categories and LIDA story topics.
4. Normalize both datasets into the same merge-ready structure.

This notebook does **not** create the final merged dataset yet.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd()
ROOT = cwd if (cwd / "data/raw").exists() else cwd.parent

LIDA_PATH = ROOT / "data/raw/lida_stories/lida_stories_en.json"
STORYWEAVER_PATH = ROOT / "data/raw/storyweaver_texts_cleaned.json"
MERGE_READY_OUTPUT_PATH = ROOT / "data/processed/lida_storyweaver_merge_ready.json"

print("Project root:", ROOT)
print("LIDA exists:", LIDA_PATH.exists(), LIDA_PATH)
print("StoryWeaver exists:", STORYWEAVER_PATH.exists(), STORYWEAVER_PATH)
print("Merge-ready output path:", MERGE_READY_OUTPUT_PATH)

## 1. Load Data

In [ ]:
with LIDA_PATH.open(encoding="utf-8") as f:
    lida = json.load(f)

with STORYWEAVER_PATH.open(encoding="utf-8") as f:
    storyweaver = json.load(f)

print("LIDA type:", type(lida).__name__)
print("LIDA records:", len(lida))
print("StoryWeaver type:", type(storyweaver).__name__)
print("StoryWeaver top-level categories:", len(storyweaver))

## 2. Inspect LIDA Dataset

LIDA is a flat list of stories. Each record already contains story text, source URL, level, attribution, and license metadata.

In [ ]:
lida_df = pd.DataFrame(lida)

print("Columns:")
print(lida_df.columns.tolist())

display(lida_df.head(3))

In [ ]:
lida_overview = lida_df[[
    "story_id", "title", "level", "word_count", "author", "source_url"
]].copy()

display(lida_overview.sort_values(["level", "story_id"]))

print("LIDA level counts:")
display(lida_df["level"].value_counts().sort_index())

print("Word count summary:")
display(lida_df["word_count"].describe())

## 3. Inspect StoryWeaver Dataset

StoryWeaver is already grouped by original content theme, then by reading level.

In [ ]:
storyweaver_rows = []

for theme, levels in storyweaver.items():
    for level_key, items in levels.items():
        for item in items:
            storyweaver_rows.append({
                "id": item.get("id"),
                "title": item.get("title"),
                "original_theme": theme,
                "level_key": level_key,
                "reading_level": item.get("reading_level"),
                "word_count": item.get("word_count"),
                "source": item.get("Source"),
                "source_url": item.get("source_url"),
                "author": item.get("author"),
                "illustrator": item.get("illustrator"),
                "text": item.get("Text"),
            })

sw_df = pd.DataFrame(storyweaver_rows)

print("StoryWeaver flattened records:", len(sw_df))
print("Columns:")
print(sw_df.columns.tolist())

display(sw_df.head(3))

In [ ]:
theme_level_counts = (
    sw_df.groupby(["original_theme", "level_key"])
    .size()
    .unstack(fill_value=0)
)
theme_level_counts["total"] = theme_level_counts.sum(axis=1)

display(theme_level_counts.sort_index())

print("StoryWeaver total:", len(sw_df))
print("Unique StoryWeaver IDs:", sw_df["id"].nunique())

print("Word count summary by level:")
display(sw_df.groupby("level_key")["word_count"].describe())

## 4. Compare Available Fields

This section checks which fields can be aligned directly and which fields need normalisation.

In [ ]:
lida_fields = set(lida_df.columns)
storyweaver_fields = set()

for theme, levels in storyweaver.items():
    for items in levels.values():
        for item in items:
            storyweaver_fields.update(item.keys())

field_comparison = pd.DataFrame({
    "field": sorted(lida_fields | storyweaver_fields),
})
field_comparison["in_lida"] = field_comparison["field"].isin(lida_fields)
field_comparison["in_storyweaver"] = field_comparison["field"].isin(storyweaver_fields)

display(field_comparison)

Initial field notes:

- LIDA text field is named `text`.
- StoryWeaver text field is named `Text`.
- LIDA ID field is `story_id`.
- StoryWeaver ID field is `id`.
- LIDA has explicit license and attribution fields.
- StoryWeaver has original content themes; LIDA does not have an original category/description field.

## 5. Normalize Into Merge-Ready Format

This section reshapes LIDA and StoryWeaver into the same columns so they can be combined with `pd.concat` later. It does not save a merged file and does not assign the future 3-class categories.

In [ ]:
MERGE_READY_COLUMNS = [
    "unified_id",
    "source_dataset",
    "source_id",
    "title",
    "text",
    "word_count",
    "reading_level",
    "reading_level_key",
    "original_category",
    "source",
    "source_url",
    "author",
    "illustrator",
    "license_name",
    "license_url",
    "attribution",
]

def normalize_lida_level_key(level):
    level_number = int(str(level).strip().split()[-1])
    if level_number <= 1:
        return "level_1"
    if level_number == 2:
        return "level_2"
    return "level_3"

lida_normalized_df = pd.DataFrame({
    "unified_id": "lida_" + lida_df["story_id"].astype(str),
    "source_dataset": "lida_stories",
    "source_id": lida_df["story_id"].astype(str),
    "title": lida_df["title"],
    "text": lida_df["text"],
    "word_count": lida_df["word_count"],
    "reading_level": lida_df["level"],
    "reading_level_key": lida_df["level"].apply(normalize_lida_level_key),
    "original_category": None,
    "source": "LIDA Stories",
    "source_url": lida_df["source_url"],
    "author": lida_df["author"],
    "illustrator": lida_df["illustrator"],
    "license_name": lida_df["license_name"],
    "license_url": lida_df["license_url"],
    "attribution": lida_df["attribution"],
})[MERGE_READY_COLUMNS]

storyweaver_normalized_df = pd.DataFrame({
    "unified_id": "storyweaver_" + sw_df["id"].astype(str),
    "source_dataset": "storyweaver",
    "source_id": sw_df["id"].astype(str),
    "title": sw_df["title"],
    "text": sw_df["text"],
    "word_count": sw_df["word_count"],
    "reading_level": sw_df["reading_level"],
    "reading_level_key": sw_df["level_key"],
    "original_category": sw_df["original_theme"],
    "source": sw_df["source"],
    "source_url": sw_df["source_url"],
    "author": sw_df["author"],
    "illustrator": sw_df["illustrator"],
    "license_name": None,
    "license_url": None,
    "attribution": None,
})[MERGE_READY_COLUMNS]

print("LIDA normalized shape:", lida_normalized_df.shape)
print("StoryWeaver normalized shape:", storyweaver_normalized_df.shape)
print("Columns match:", list(lida_normalized_df.columns) == list(storyweaver_normalized_df.columns))

display(lida_normalized_df.head())
display(storyweaver_normalized_df.head())

In [ ]:
merge_ready_df = pd.concat(
    [storyweaver_normalized_df, lida_normalized_df],
    ignore_index=True,
)

print("LIDA records:", len(lida_normalized_df))
print("StoryWeaver records:", len(storyweaver_normalized_df))
print("Merge-ready preview records:", len(merge_ready_df))

display(merge_ready_df.head())
display(merge_ready_df.tail())

## 6. Validation Checks

These checks verify the merge-ready structure, missing values, LIDA `reading_level_key` conversion, and possible duplicate stories across LIDA and StoryWeaver. LIDA's original `reading_level` values stay unchanged, including `Level 4` and `Level 5`. Suspected duplicates are displayed only; nothing is removed here.

In [ ]:
print("Unified IDs are unique:", merge_ready_df["unified_id"].is_unique)

source_id_duplicates = (
    merge_ready_df.groupby("source_dataset")["source_id"]
    .apply(lambda values: values.duplicated().sum())
    .rename("duplicate_source_id_count")
)
display(source_id_duplicates)

missing_counts = merge_ready_df[["title", "text", "word_count", "reading_level_key"]].isna().sum()
display(missing_counts.rename("missing_count"))

lida_level_check = (
    lida_normalized_df.groupby(["reading_level", "reading_level_key"])
    .size()
    .reset_index(name="count")
    .sort_values(["reading_level", "reading_level_key"])
)
display(lida_level_check)

In [ ]:
# Cross-source duplicate check: LIDA stories vs StoryWeaver stories.
# This checks exact normalized title/text matches first, then uses TF-IDF cosine
# similarity to surface highly similar story pairs for manual review.

def normalize_for_duplicate_check(series):
    return (
        series.fillna("")
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )

duplicate_check_df = merge_ready_df.copy()
duplicate_check_df["title_normalized"] = normalize_for_duplicate_check(duplicate_check_df["title"])
duplicate_check_df["text_normalized"] = normalize_for_duplicate_check(duplicate_check_df["text"])

cross_source_title_dupes = duplicate_check_df[
    duplicate_check_df["title_normalized"].ne("")
    & duplicate_check_df.groupby("title_normalized")["source_dataset"].transform("nunique").gt(1)
].sort_values(["title_normalized", "source_dataset", "source_id"])

cross_source_text_dupes = duplicate_check_df[
    duplicate_check_df["text_normalized"].ne("")
    & duplicate_check_df.groupby("text_normalized")["source_dataset"].transform("nunique").gt(1)
].sort_values(["text_normalized", "source_dataset", "source_id"])

duplicate_display_columns = ["source_dataset", "source_id", "title", "word_count"]

print("Exact cross-source duplicate titles:", len(cross_source_title_dupes))
display(cross_source_title_dupes[duplicate_display_columns])

print("Exact cross-source duplicate texts:", len(cross_source_text_dupes))
display(cross_source_text_dupes[duplicate_display_columns])

lida_for_similarity = duplicate_check_df[duplicate_check_df["source_dataset"] == "lida_stories"].reset_index(drop=True)
storyweaver_for_similarity = duplicate_check_df[duplicate_check_df["source_dataset"] == "storyweaver"].reset_index(drop=True)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(
    pd.concat([
        lida_for_similarity["text_normalized"],
        storyweaver_for_similarity["text_normalized"],
    ], ignore_index=True)
)

lida_matrix = tfidf_matrix[:len(lida_for_similarity)]
storyweaver_matrix = tfidf_matrix[len(lida_for_similarity):]
similarity_matrix = cosine_similarity(lida_matrix, storyweaver_matrix)

similarity_rows = []
for lida_index, similarities in enumerate(similarity_matrix):
    best_storyweaver_index = similarities.argmax()
    best_score = similarities[best_storyweaver_index]
    lida_row = lida_for_similarity.iloc[lida_index]
    storyweaver_row = storyweaver_for_similarity.iloc[best_storyweaver_index]
    similarity_rows.append({
        "similarity_score": round(float(best_score), 4),
        "lida_id": lida_row["source_id"],
        "lida_title": lida_row["title"],
        "lida_word_count": lida_row["word_count"],
        "storyweaver_id": storyweaver_row["source_id"],
        "storyweaver_title": storyweaver_row["title"],
        "storyweaver_word_count": storyweaver_row["word_count"],
    })

similarity_review_df = pd.DataFrame(similarity_rows).sort_values("similarity_score", ascending=False)
potential_cross_source_duplicates = similarity_review_df[similarity_review_df["similarity_score"] >= 0.80]

print("Potential cross-source duplicates by text similarity >= 0.80:", len(potential_cross_source_duplicates))
display(potential_cross_source_duplicates)

print("Top 10 closest LIDA-to-StoryWeaver text matches for review:")
display(similarity_review_df.head(10))

## 7. Data Quality Checks

Before saving the merge-ready dataset, check for issues that could affect later classification: very short texts, duplicate titles within a source, and large differences between stored `word_count` and a simple recalculated word count.

In [ ]:
quality_df = merge_ready_df.copy()
quality_df["recomputed_word_count"] = quality_df["text"].fillna("").str.findall(r"\b[\w'-]+\b").str.len()
quality_df["word_count_diff"] = quality_df["word_count"] - quality_df["recomputed_word_count"]
quality_df["word_count_relative_diff"] = (
    quality_df["word_count_diff"].abs() / quality_df["word_count"].replace(0, pd.NA)
)
quality_df["large_word_count_mismatch"] = (
    quality_df["word_count_diff"].abs().gt(20)
    & quality_df["word_count_relative_diff"].gt(0.25)
)
quality_df["title_normalized"] = normalize_for_duplicate_check(quality_df["title"])

quality_summary = (
    quality_df.groupby("source_dataset")
    .agg(
        records=("unified_id", "count"),
        min_words=("word_count", "min"),
        median_words=("word_count", "median"),
        max_words=("word_count", "max"),
        very_short_texts=("word_count", lambda values: (values < 20).sum()),
        large_word_count_mismatches=("large_word_count_mismatch", "sum"),
    )
    .reset_index()
)

display(quality_summary)

In [ ]:
very_short_texts = quality_df[quality_df["word_count"] < 20].sort_values(
    ["source_dataset", "word_count", "title"]
)

print("Very short texts under 20 words:", len(very_short_texts))
display(very_short_texts[[
    "source_dataset", "source_id", "title", "word_count", "reading_level", "original_category"
]])

In [ ]:
within_source_duplicate_titles = quality_df[
    quality_df["title_normalized"].ne("")
    & quality_df.groupby(["source_dataset", "title_normalized"])["unified_id"].transform("count").gt(1)
].sort_values(["source_dataset", "title_normalized", "source_id"])

print("Within-source duplicate titles:", len(within_source_duplicate_titles))
display(within_source_duplicate_titles[[
    "source_dataset", "source_id", "title", "word_count", "reading_level", "original_category"
]])

In [ ]:
large_word_count_mismatches = quality_df[quality_df["large_word_count_mismatch"]].sort_values(
    ["source_dataset", "word_count_relative_diff"], ascending=[True, False]
)

print("Large word count mismatches:", len(large_word_count_mismatches))
display(large_word_count_mismatches[[
    "source_dataset", "source_id", "title", "word_count", "recomputed_word_count", "word_count_diff", "word_count_relative_diff"
]].head(30))

## 8. Save Merge-Ready Dataset

The saved file keeps the unified structure only. It does not include the future 3-class labels. Classification should happen in a separate notebook after deciding the category rules. Quality-check results stay in this notebook rather than a separate summary file.

In [ ]:
MERGE_READY_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

merge_ready_records = merge_ready_df[MERGE_READY_COLUMNS].to_dict(orient="records")
MERGE_READY_OUTPUT_PATH.write_text(
    json.dumps(merge_ready_records, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved merge-ready dataset:", MERGE_READY_OUTPUT_PATH)
print("Records saved:", len(merge_ready_records))